# 04. XAI Data Attribution: Influence Functions, TracIn, and TRAK

- **Category:** 10_xai_and_ai_analysis_methods
- **Lineage & Citations:** Cook & Weisberg (1982 *Residuals and Influence in Regression*); Koh & Liang (2017 *Understanding Black-box Predictions via Influence Functions*); Ghorbani & Zou (2019 *Data Shapley*); Pruthi et al. (2020 *Estimating Training Data Influence by Tracing Gradient Descent: TracIn*); Park et al. (2023 *TRAK: Attributing Model Behavior at Scale*).
- **Purpose:** Attribute black-box test predictions back to specific training samples using inverse Hessian-vector products (Influence Functions), training trajectory checkpoint dot products (TracIn), and random projection features (TRAK).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [03. XAI Concepts TCAV CBM Counterfactuals](03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.ipynb) $\to$ `🔗 Downstream Syntheses:` [05. Mechanistic Interpretability Probing Lens Patching SAE](05_mechanistic_interpretability_probing_lens_patching_and_sae.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Avoid undamped influence functions on non-convex deep models where empirical Hessians possess negative eigenvalues (causing LiSSA or conjugate gradient solvers to diverge); do not run exact $2^N$ Data Shapley on large datasets (use TracIn or TRAK instead).

## Mathematical Foundations & Governing Equations

1. **Influence Function (IHVP)**: $\mathcal{I}(z_{\text{train}}, z_{\text{test}}) = - \nabla_\theta \mathcal{L}(z_{\text{train}})^T (H + \lambda I)^{-1} \nabla_\theta \mathcal{L}(z_{\text{test}})$
2. **TracIn Trajectory Attribution**: $\text{TracIn}(z_{\text{train}}, z_{\text{test}}) = \sum_{t=1}^T \eta_t \left( \nabla \mathcal{L}(z_{\text{train}}, \theta_t) \cdot \nabla \mathcal{L}(z_{\text{test}}, \theta_t) \right)$
3. **TRAK Projected Attribution**: $\phi(z) = \nabla_\theta \mathcal{L}(z) P, \quad S_i = \frac{\phi(z_i) \cdot \phi(z_{\text{test}})}{\|\phi(z_i)\|_2 \|\phi(z_{\text{test}})\|_2 + \epsilon}$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/10_xai_and_ai_analysis_methods/04_xai_data_attribution_influence_functions_tracin_and_trak.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_influence(hessian, g_test, g_train, damping=1e-3):
    d = hessian.shape[0]
    h_damped = hessian + damping * torch.eye(d, dtype=hessian.dtype, device=hessian.device)
    ihvp = torch.linalg.solve(h_damped, g_test)
    inf = -(g_train * ihvp).sum()
    return ihvp, inf

def calc_tracin(train_g, test_g, lrs):
    dots = (train_g * test_g).sum(dim=-1)
    return (lrs * dots).sum()

def calc_trak(train_g, test_g, proj):
    phi_tr = train_g @ proj
    phi_te = test_g @ proj
    norm_tr = torch.sqrt((phi_tr * phi_tr).sum(dim=-1, keepdim=True) + 1e-8)
    norm_te = torch.sqrt((phi_te * phi_te).sum() + 1e-8)
    return ((phi_tr @ phi_te.unsqueeze(-1)) / (norm_tr * norm_te)).squeeze(-1)

In [ ]:
# Parity verification against golden reference tensors
ihvp, inf_score = calc_influence(
    weights["inf_hessian"], weights["inf_grad_test"], weights["inf_grad_train"], damping=1e-3
)
assert torch.max(torch.abs(ihvp - weights["inf_ihvp_ref"])) < 1e-5, "Influence IHVP mismatch"
assert torch.max(torch.abs(inf_score - weights["inf_score_ref"])) < 1e-5, "Influence score mismatch"

tracin = calc_tracin(weights["tracin_train_g"], weights["tracin_test_g"], weights["tracin_lrs"])
assert torch.max(torch.abs(tracin - weights["tracin_score_ref"])) < 1e-5, "TracIn mismatch"

trak = calc_trak(weights["trak_train_g"], weights["trak_test_g"], weights["trak_proj"])
assert torch.max(torch.abs(trak - weights["trak_scores_ref"])) < 1e-5, "TRAK mismatch"
print("All Influence, TracIn, and TRAK assertions passed!")

In [ ]:
# Benchmarking data attribution operators
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_influence(weights["inf_hessian"], weights["inf_grad_test"], weights["inf_grad_train"], damping=1e-3)
    _ = calc_tracin(weights["tracin_train_g"], weights["tracin_test_g"], weights["tracin_lrs"])
    _ = calc_trak(weights["trak_train_g"], weights["trak_test_g"], weights["trak_proj"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused Influence + TracIn + TRAK latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Undamped Hessian Inversion Instability on Indefinite Curvature
# Near-singular or indefinite Hessian causes inversion explosion
h_ill = torch.tensor([[1.0, 1.0], [1.0, 1.0001]])
g_sample = torch.tensor([1.0, -1.0])
# Damped IHVP is stable
v_damped = torch.linalg.solve(h_ill + 0.1 * torch.eye(2), g_sample)
# Undamped IHVP explodes by orders of magnitude
v_undamped = torch.linalg.solve(h_ill + 1e-7 * torch.eye(2), g_sample)
print(f"Damped norm: {torch.norm(v_damped):.2f} vs Undamped norm: {torch.norm(v_undamped):.2f}")
assert torch.norm(v_undamped) > 100.0 * torch.norm(v_damped), "Undamped Hessian inversion explodes!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Influence Functions** identify influential training points via infinitesimal upweighting and inverse Hessian-vector products.
2. **TracIn** traces gradient descent checkpoints to accumulate empirical sample contributions without second-order derivatives.
3. **TRAK** scales gradient attribution to large models using random Johnson-Lindenstrauss projections.
4. **Damping** is essential to prevent Hessian inversion explosion on near-singular or indefinite loss surfaces.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_xai_data_attribution_influence_functions_tracin_and_trak.md](../../markdown/10_xai_and_ai_analysis_methods/04_xai_data_attribution_influence_functions_tracin_and_trak.md)
- **PyTorch 2.x Module:** [04_xai_data_attribution_influence_functions_tracin_and_trak.py](../../pytorch2/10_xai_and_ai_analysis_methods/04_xai_data_attribution_influence_functions_tracin_and_trak.py)
- **Functional JAX Module:** [04_xai_data_attribution_influence_functions_tracin_and_trak.py](../../jax/10_xai_and_ai_analysis_methods/04_xai_data_attribution_influence_functions_tracin_and_trak.py)
- **Native Rust Candle Track:** [04_xai_data_attribution_influence_functions_tracin_and_trak.rs](../../candle/10_xai_and_ai_analysis_methods/04_xai_data_attribution_influence_functions_tracin_and_trak.rs)
- **Hardware Kernel (CPU / WGSL):** [04_xai_data_attribution_influence_functions_tracin_and_trak.rs](../../kernels/10_xai_and_ai_analysis_methods/04_xai_data_attribution_influence_functions_tracin_and_trak.rs)
- **Golden SafeTensors:** [04_xai_data_attribution_influence_functions_tracin_and_trak.safetensors](../../weights/10_xai_and_ai_analysis_methods/04_xai_data_attribution_influence_functions_tracin_and_trak.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. XAI Concepts TCAV CBM Counterfactuals](03_xai_concepts_tcav_cbm_counterfactuals_dice_and_global_ale.ipynb) | [Master Index](../../README.md#10_xai_and_ai_analysis_methods) | [Next: 05. Mechanistic Interpretability Probing Lens Patching SAE](05_mechanistic_interpretability_probing_lens_patching_and_sae.ipynb) &rarr;